In [1]:
!pip install psycopg2-binary sqlalchemy

In [2]:
import pandas as pd
from sqlalchemy import create_engine, inspect

# параметры подключения к PostgreSQL
username = 'postgres'   # Имя пользователя
password = '3441'   # Пароль
host = 'localhost'           # Хост (например, 'localhost' или IP адрес)
port = '5432'                # Порт (по умолчанию для PostgreSQL — 5432)
database = 'my_database'   # Имя базы данных

# Создаем SQLAlchemy engine для PostgreSQL
engine = create_engine(f'postgresql://{username}:{password}@{host}:{port}/{database}', echo=False)

# Загрузим CSV файлы в DataFrame
df1 = pd.read_csv('/Users/sergey/Desktop/ТЗ Корона/sql/t_agent.csv', sep=';')
df2 = pd.read_csv('/Users/sergey/Desktop/ТЗ Корона/sql/t_city.csv', sep=';')
df3 = pd.read_csv('/Users/sergey/Desktop/ТЗ Корона/sql/t_country.csv', sep=';')
df4 = pd.read_csv('/Users/sergey/Desktop/ТЗ Корона/sql/t_service.csv', sep=';')
df5 = pd.read_csv('/Users/sergey/Desktop/ТЗ Корона/sql/t_transfer.csv', sep=';')

# Добавляем каждую таблицу в базу данных PostgreSQL
df1.to_sql('t_agent', con=engine, index=False, if_exists='replace')
df2.to_sql('t_city', con=engine, index=False, if_exists='replace')
df3.to_sql('t_country', con=engine, index=False, if_exists='replace')
df4.to_sql('t_service', con=engine, index=False, if_exists='replace')
df5.to_sql('t_transfer', con=engine, index=False, if_exists='replace')

# Проверим, что таблицы были добавлены
inspector = inspect(engine)
tables = inspector.get_table_names()
print(tables)


['t_agent', 't_city', 't_country', 't_service', 't_transfer']


Написал функцию, для просмотра таблиц, что бы проверить на правильность выгрузки

In [3]:
def open_table(name):
    # Выполним SQL-запрос с использованием f-строки для подстановки имени таблицы
    query = f'''
    SELECT * 
    FROM {name}
    '''
    result = pd.read_sql(query, con=engine)

    # Возвращаем результат
    return result

In [4]:
# Открыл при помощи SQL запроса таблицы t_agent
open_table('t_agent')

,agent_ID,fullname,work_status,city_id
0,AG00014,"ООО КБ ""Банк первый"", отделение 1",ACTIVE,1
1,AG00015,"ООО КБ ""Банк первый"", отделение 2",ACTIVE,5
2,AG00016,"ООО КБ ""Банк первый"", отделение 3",ACTIVE,100
3,AG00017,"ООО КБ ""Банк первый"", отделение 4",ACTIVE,100
4,AG00018,"ОАО КБ ""Банк малый"", филиал 1",CLOSED,5
5,AG00019,"ОАО КБ ""Банк малый"", филиал 2",CLOSED,5
6,AG00020,"ОАО КБ ""Банк малый"", филиал 3",ACTIVE,12
7,AG00021,"ОАО КБ ""Банк малый"", филиал 4",ACTIVE,13
8,AG00022,"ОАО КБ ""Банк малый"", филиал 5",ACTIVE,1
9,AG00023,"ОАО КБ ""Банк малый"", филиал 6",ACTIVE,1


In [5]:
# Открыл при помощи SQL запроса таблицы t_city
open_table('t_city')

,city_id,city_name,country_id
0,1,Новосибирск,RUS
1,5,Москва,RUS
2,100,Екатеринбург,RUS
3,12,Томcк,RUS
4,13,Омск,RUS
5,9777,Киев,UKR


In [6]:
# Открыл при помощи SQL запроса таблицы t_country
open_table('t_country')

,country_id,country_name
0,RUS,Россия
1,UKR,Украина
2,GEO,Грузия
3,UZB,Узбекистан
4,TJK,Таджикистан


In [7]:
# Открыл при помощи SQL запроса таблицы t_service
open_table('t_service')

,agent_id,service,cur
0,AG00014,Услуга отправки,RUB
1,AG00014,Услуга выдачи,RUB
2,AG00015,Услуга отправки,RUB
3,AG00015,Услуга выдачи,RUB
4,AG00016,Услуга отправки,RUB
5,AG00016,Услуга выдачи,RUB
6,AG00017,Услуга отправки,RUB
7,AG00017,Услуга выдачи,RUB
8,AG00018,Услуга отправки,RUB
9,AG00018,Услуга выдачи,RUB


In [8]:
# Открыл при помощи SQL запроса таблицы t_transfer
open_table('t_transfer')

,transfer_number,amount,cur,send_date,receive_date,status,send_to_city_id,bank_payer,bank_payee,payer_fullname,payee_fullname
0,436573674,5000,RUR,6/1/19,None,Отправлен,9777,AG00019,None,АЛЕКСЕЕНКО ЕВГЕНИЙ ВАЛЕРЬЕВИЧ,АМАНКУЛОВ САЛМООРБЕК КАЛДАРБЕКОВИЧ
1,473472123,100000,RUR,6/1/19,None,Отправлен,1,AG00026,None,АХМЕРОВ МАРАТ ХАЛИЛОВИЧ,МУСАЕВА АНАРКАН ТОКТОГУЛОВНА
2,306556689,12000,RUR,6/2/19,None,Отправлен,5,AG00023,None,ЛОМИНЕИШВИЛИ ТАМАЗИ ИВАНОВИЧ,САДИРОВА ЭЛЬМИРА ТОКТОБАЕВНА
3,967353535,4000,RUR,6/1/19,None,Отправлен,1,AG00028,None,МАЛЬГИН МАКСИМ СЕРГЕЕВИЧ,АБДИКАДИРОВ АБДИГАНИ НЕТ
4,211244435,1900,RUR,6/2/19,6/4/19,Выдан,1,AG00025,AG00014,МЕЛЬНИКОВА ЗОЯ АЛЕКСАНДРОВНА,МОЩЕНКО ИРИНА ВЛАДИМИРОВНА
5,356344466,40000,RUR,6/2/19,6/5/19,Выдан,5,AG00019,AG00015,ОСЕЕВА ОКСАНА АЛЕКСАНДРОВНА,ХОДЖАЕВ ИХТИЁР АХРОРОВИЧ
6,379598916,2000,USD,6/2/19,None,Отправлен,9777,AG00019,None,ОСЕЕВА ОКСАНА АЛЕКСАНДРОВНА,БОРИСОВ СЕРГЕЙ АЛЕКСЕЕВИЧ
7,338295266,4000,RUR,6/3/19,6/4/19,Выдан,5,AG00018,AG00018,ОСЕЕВА ОКСАНА АЛЕКСАНДРОВНА,УСУБАЛИЕВА АЙНАГУЛ АПЫНОВНА
8,214384316,22200,RUR,6/2/19,6/5/19,Выдан,5,AG00015,AG00019,РАЧЕК ДМИТРИЙ СЕРГЕЕВИЧ,БОРИСОВ СЕРГЕЙ АЛЕКСЕЕВИЧ
9,879346569,1000,RUR,6/1/19,6/5/19,Выдан,100,AG00028,AG00028,СОЛОВЬЕВ АНДРЕЙ АНАТОЛЬЕВИЧ,ПИЖАМОВ АНДРЕЙ АЛЕКСАНДРОВИЧ


Описание:

Имеются следующие таблицы базы данных системы денежных переводов:
- Справочник городов (t_city).
- Справочник стран (t_country).
- Справочник агентов (t_agent) - это банки, которые отправляют/выдают денежные переводы. Банк на текущий момент времени может быть действующим (ACTIVE) или отключенным (CLOSED). Отключенный от системы банк не может выдать денежный перевод.
- Справочник настроек агентов (t_service). При наличие записи(-ей) определяет для конкретного агента техническую возможность отправлять или выдавать денежные переводы в той или иной валюте.
- Операции (t_transfer). Каждая операция представляет собой денежный перевод, совершенный клиентом-физ.лицом в одном из банков. 
У перевода есть: уникальный номер; сумма; валюта; имя клиента-отправителя перевода; имя клиента-получателя, указанное отправителем; банк и дата отправки перевода; банк и дата выдачи перевода (если перевод уже выдан); статус перевода (отправлен или уже выдан); город, указанный клиентом-отправителем, куда совершен денежный перевод (по условиям задачи подразумевается, что выдача перевода получателю возможна только в пределах банков этого же города)

**Задание1:**
- В целях получения статистики, показывающей эффективность проведенной в мае стимулирующей кампании, 
реализовать sql-скрипт, подсчитывающий оборот отправленных денежных переводов банками РФ за первую декаду июня в каждой из валют. 
- Формат: Сумма оборота, валюта.

In [9]:
query = '''
SELECT SUM(amount) AS всего,
    cur AS валюта
FROM t_transfer AS t
JOIN t_agent AS a ON t.bank_payer = a."agent_ID"
JOIN t_city AS c ON a.city_id = c.city_id
WHERE DATE(t.send_date) BETWEEN '2019-06-01' AND '2019-06-10'
    AND t.status = 'Отправлен'
    AND c.country_id = 'RUS'
GROUP BY cur
'''

result = pd.read_sql(query, con=engine)
result

,всего,валюта
0,121000.0,RUR
1,3000.0,USD


**Задание2:**    
В целях предоставления подробного отчета менеджеру компании, реализовать SQL-скрипт, выводящий подробную информацию об отправленных платежах за июнь:
1. дата отправки в формате  "ГОД/МЕСЯЦ", сумма отправленного перевода, валюта отправленного перевода, 
2. город банка отправителя, страна банка отправителя, наименование банка отправителя,
3. город банка получателя, страна банка получателя, наименование банка получателя

In [10]:
query ='''
SELECT 
    TO_CHAR(TO_DATE(t.send_date, 'MM/DD/YY'), 'YYYY/MM') AS дата_отправления,
    t.amount AS сумма_перевода,
    t.cur AS валюта,
    sender_city.city_name AS город_отправителя,
    sender_country.country_name AS страна_отправителя,
    sender_agent.fullname AS банк_отправителя,
    receiver_city.city_name AS город_получения,
    receiver_country.country_name AS страна_получения,
    receiver_agent.fullname AS банк_получателя
FROM 
    t_transfer AS t
JOIN 
    t_agent AS sender_agent ON t.bank_payer = sender_agent."agent_ID"
JOIN 
    t_city AS sender_city ON sender_agent.city_id = sender_city.city_id
JOIN 
    t_country AS sender_country ON sender_city.country_id = sender_country.country_id
JOIN 
    t_city AS receiver_city ON t.send_to_city_id = receiver_city.city_id
JOIN 
    t_country AS receiver_country ON receiver_city.country_id = receiver_country.country_id
LEFT JOIN 
    t_agent AS receiver_agent ON t.bank_payee = receiver_agent."agent_ID"
WHERE EXTRACT(MONTH FROM TO_DATE(t.send_date, 'MM/DD/YY')) = 6
    AND t.status = 'Отправлен'
'''
    
result = pd.read_sql(query, con=engine)
result

,дата_отправления,сумма_перевода,валюта,город_отправителя,страна_отправителя,банк_отправителя,город_получения,страна_получения,банк_получателя
0,2019/06,2000,USD,Москва,Россия,"ОАО КБ ""Банк малый"", филиал 2",Киев,Украина,None
1,2019/06,5000,RUR,Москва,Россия,"ОАО КБ ""Банк малый"", филиал 2",Киев,Украина,None
2,2019/06,1000,USD,Томcк,Россия,"ОАО КБ ""Банк малый"", филиал 3",Киев,Украина,None
3,2019/06,12000,RUR,Новосибирск,Россия,"ОАО КБ ""Банк малый"", филиал 6",Москва,Россия,None
4,2019/06,4000,RUR,Екатеринбург,Россия,"ЗАО КБ ""Банк большой"", ОКВКО 5",Новосибирск,Россия,None
5,2019/06,100000,RUR,Томcк,Россия,"ЗАО КБ ""Банк большой"", ОКВКО 3",Новосибирск,Россия,None
6,2019/06,70000,RUR,Киев,Украина,"ЗАО КБ ""Банк большой"", ОКВКО 1",Омск,Россия,None


**Задание3:**   
В целях проведения кампании по рассылке рекламных СМС-уведомлений клиентам системы, реализовать sql-скрипт, выводящий список ФИО клиентов-отправителей, совершивших за июнь более двух платежей, а также кол-во и общую сумму отправленных каждым из них платежей. 
Список отсортировать по полю "Кол-во платежей" в обратном порядке (от большего к меньшему).

In [11]:
query = '''
SELECT 
    payer_fullname AS отправитель,
    COUNT(transfer_number) AS кол_во_платежей,
    SUM(amount) AS общая_сумма
FROM 
    t_transfer
WHERE 
    EXTRACT(MONTH FROM TO_DATE(send_date, 'MM/DD/YY')) = 6 
GROUP BY 
    payer_fullname
HAVING 
    COUNT(transfer_number) > 2 
ORDER BY 
    кол_во_платежей DESC; 
'''

result = pd.read_sql(query, con=engine)
result

,отправитель,кол_во_платежей,общая_сумма
0,ОСЕЕВА ОКСАНА АЛЕКСАНДРОВНА,3,46000.0
1,СОЛОВЬЕВ АНДРЕЙ АНАТОЛЬЕВИЧ,3,48300.0
